In [1]:
import pandas as pd
import sqlite3

# 1. Cargar los archivos CSV
dim_equipos = pd.read_csv('dim_equipos.csv')
fact_npt_fallas = pd.read_csv('fact_npt_fallas.csv')
fact_parametros = pd.read_csv('fact_parametros_operativos.csv')
dim_pozos = pd.read_csv('dim_pozos.csv')

# 2. Crear una base de datos SQLite temporal en la memoria RAM
conn = sqlite3.connect(':memory:')

# 3. Convertir los DataFrames de Pandas en tablas SQL dentro de la base temporal
dim_equipos.to_sql('dim_equipos', conn, index=False, if_exists='replace')
fact_npt_fallas.to_sql('fact_npt_fallas', conn, index=False, if_exists='replace')
fact_parametros.to_sql('fact_parametros_operativos', conn, index=False, if_exists='replace')
dim_pozos.to_sql('dim_pozos', conn, index=False, if_exists='replace')

5

In [2]:
# ==========================================
# QUERY 1: IMPACTO OPERATIVO (NPT POR EQUIPO)
# ==========================================
query_1 = """
SELECT 
    e.tipo_equipo,
    e.modelo,
    COUNT(f.falla_id) AS cantidad_eventos_falla,
    ROUND(SUM(f.horas_npt), 1) AS total_horas_perdidas,
    ROUND(AVG(f.horas_npt), 2) AS promedio_horas_por_falla
FROM fact_npt_fallas f
JOIN dim_equipos e 
    ON f.equipo_id = e.equipo_id
GROUP BY 
    e.tipo_equipo, 
    e.modelo
ORDER BY 
    total_horas_perdidas DESC;
"""

resultado_query_1 = pd.read_sql_query(query_1, conn)
print("--- Impacto Operativo: Total de Horas NPT por Equipo ---")
display(resultado_query_1)

--- Impacto Operativo: Total de Horas NPT por Equipo ---


,tipo_equipo,modelo,cantidad_eventos_falla,total_horas_perdidas,promedio_horas_por_falla
0,Zaranda Secundaria,Derrick Hyperpool,62,221.2,3.57
1,Zaranda Primaria,Derrick Hyperpool,62,212.9,3.43
2,Zaranda Secundaria,Swaco MONGOOSE,41,152.3,3.71
3,Zaranda Primaria,NOV Brandt KING COBRA,15,55.8,3.72
4,Centrífuga Decantadora,Derrick 7200,14,42.4,3.03
5,Deslimador,Nov Brandt 16 conos,9,39.4,4.38
6,Deslimador,Derrick 20 conos,9,32.3,3.59
7,Desarenador,Nov Brandt 2 conos,5,22.2,4.44
8,Desarenador,Derrick 3 conos,4,16.8,4.20
9,Centrífuga Decantadora,Derrick DE-1000,2,6.7,3.35


In [3]:
# ==========================================
# QUERY 2: ANÁLISIS DE CAUSA RAÍZ EN ZARANDAS
# ==========================================
query_2 = """
SELECT 
    p.tipo_lodo,
    p.malla_api,
    COUNT(DISTINCT p.fecha_hora) AS turnos_operados,
    COUNT(DISTINCT f.falla_id) AS eventos_falla,
    ROUND(COUNT(DISTINCT f.falla_id) * 100.0 / NULLIF(COUNT(DISTINCT p.fecha_hora), 0), 2) AS porcentaje_incidencia_falla
FROM fact_parametros_operativos p
JOIN dim_equipos e 
    ON p.equipo_id = e.equipo_id
LEFT JOIN fact_npt_fallas f 
    ON p.equipo_id = f.equipo_id 
    AND p.fecha_hora = f.fecha_hora
WHERE 
    e.tipo_equipo LIKE 'Zaranda%'
GROUP BY 
    p.tipo_lodo, 
    p.malla_api
ORDER BY 
    porcentaje_incidencia_falla DESC;
"""

resultado_query_2 = pd.read_sql_query(query_2, conn)
print("--- Análisis de Causa Raíz (Lodo vs Malla) ---")
display(resultado_query_2.head(10)) # Muestra los top 10 más críticos

--- Análisis de Causa Raíz (Lodo vs Malla) ---


,tipo_lodo,malla_api,turnos_operados,eventos_falla,porcentaje_incidencia_falla
0,OBM,230,260,59,22.69
1,OBM,170,249,51,20.48
2,OBM,200,272,54,19.85
3,OBM,140,260,6,2.31
4,OBM,100,271,5,1.85
5,WBM,100,204,3,1.47
6,WBM,200,208,1,0.48
7,WBM,170,214,1,0.47
8,WBM,140,209,0,0.00
9,WBM,230,207,0,0.00


In [4]:
# ==========================================
# QUERY 3: CÁLCULO DE MTBF (MEAN TIME BETWEEN FAILURES)
# ==========================================
query_3 = """
WITH horas_operativas AS (
    SELECT 
        equipo_id,
        COUNT(fecha_hora) * 6 AS horas_totales_trabajadas 
    FROM fact_parametros_operativos
    GROUP BY equipo_id
),
fallas_equipo AS (
    SELECT 
        equipo_id,
        COUNT(falla_id) AS numero_fallas
    FROM fact_npt_fallas
    GROUP BY equipo_id
)
SELECT 
    e.equipo_id,
    e.tipo_equipo,
    e.modelo,
    ho.horas_totales_trabajadas,
    COALESCE(fe.numero_fallas, 0) AS total_fallas,
    ROUND(ho.horas_totales_trabajadas * 1.0 / NULLIF(fe.numero_fallas, 0), 2) AS mtbf_horas
FROM dim_equipos e
JOIN horas_operativas ho 
    ON e.equipo_id = ho.equipo_id
LEFT JOIN fallas_equipo fe 
    ON e.equipo_id = fe.equipo_id
ORDER BY 
    mtbf_horas ASC; -- Ordenamos de menor a mayor MTBF (los que fallan más rápido arriba)
"""

resultado_query_3 = pd.read_sql_query(query_3, conn)
print("\n--- MTBF (Tiempo Medio Entre Fallas) ---")
display(resultado_query_3.head(10))


--- MTBF (Tiempo Medio Entre Fallas) ---


,equipo_id,tipo_equipo,modelo,horas_totales_trabajadas,total_fallas,mtbf_horas
0,EQ-008,Desarenador,Nov Brandt 2 conos,2190,0,NaN
1,EQ-018,Desarenador,Nov Brandt 2 conos,2190,0,NaN
2,EQ-017,Zaranda Secundaria,Derrick Hyperpool,2190,26,84.23
3,EQ-007,Zaranda Secundaria,Derrick Hyperpool,2190,24,91.25
4,EQ-012,Zaranda Secundaria,Swaco MONGOOSE,2190,22,99.55
5,EQ-002,Zaranda Secundaria,Swaco MONGOOSE,2190,19,115.26
6,EQ-011,Zaranda Primaria,Derrick Hyperpool,2190,18,121.67
7,EQ-016,Zaranda Primaria,Derrick Hyperpool,2190,17,128.82
8,EQ-001,Zaranda Primaria,Derrick Hyperpool,2190,15,146.00
9,EQ-021,Zaranda Primaria,NOV Brandt KING COBRA,2190,15,146.00


In [5]:
# ==========================================
# QUERY 4: IMPACTO GEOGRÁFICO Y POR TIPO DE POZO
# ==========================================
query_4 = """
SELECT 
    p.bloque_operativo,
    p.pozo_id,
    p.tipo_perforacion,
    COUNT(f.falla_id) AS eventos_falla,
    ROUND(SUM(f.horas_npt), 1) AS total_horas_perdidas
FROM fact_npt_fallas f
JOIN dim_equipos e 
    ON f.equipo_id = e.equipo_id
JOIN dim_pozos p 
    ON e.pozo_id = p.pozo_id
GROUP BY 
    p.bloque_operativo, 
    p.pozo_id,
    p.tipo_perforacion
ORDER BY 
    total_horas_perdidas DESC;
"""

resultado_query_4 = pd.read_sql_query(query_4, conn)
print("--- NPT Distribuido por Bloque y Pozo ---")
display(resultado_query_4)

--- NPT Distribuido por Bloque y Pozo ---


,bloque_operativo,pozo_id,tipo_perforacion,eventos_falla,total_horas_perdidas
0,Valle Medio del Magdalena,PZ-003,Vertical,50,178.6
1,Llanos - Paz de Ariporo,PZ-001,Direccional,42,162.4
2,Cuenca Putumayo,PZ-004,Direccional,48,159.2
3,Valle Medio del Magdalena,PZ-005,Vertical,39,153.6
4,Llanos - Paz de Ariporo,PZ-002,Horizontal,44,148.2
